# Week 49, Day 4: Timed set 2

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w049d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload or close the page. It doesn't stop you working when time is up: that's on your honour..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: B1: Products of the others

Write `products_except_self(nums)`: a list where item i is the product of every number in `nums` except `nums[i]`, **without division** (zeros are allowed) and in O(n), using a pass of products from the left and a pass from the right.

In [ ]:
def products_except_self(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, math\ndef test_products():\n    \"products\"\n    assert products_except_self([1, 2, 3, 4]) == [24, 12, 8, 6] and products_except_self([0, 4, 0]) == [0, 0, 0]\n    assert products_except_self([2, 0, 3]) == [0, 6, 0] and products_except_self([5]) == [1] and products_except_self([]) == []\n    rng = random.Random(1)\n    for _ in range(200):\n        xs = [rng.randint(-3, 3) for _ in range(rng.randint(0, 8))]\n        assert products_except_self(xs) == [math.prod(xs[:i] + xs[i + 1:]) for i in range(len(xs))]\n    assert \"/\" not in source.split(\"def products_except_self\")[1]")

### Exercise: B2: A least-recently-used cache

Write `LRUCache(capacity)` with `get(key)` (the value, or `None`; a successful get makes the key the most recently used) and `put(key, value)` (insert or update, making it most recently used; if this makes the cache exceed its capacity, evict the **least** recently used key). Also `keys()`: the keys from least to most recently used. Every operation except `keys` must be O(1). You may build it from a dict and a doubly linked list (Week 42), or use `collections.OrderedDict`, whose `move_to_end` and `popitem(last=False)` are O(1).

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_cache():\n    \"the cache\"\n    c = LRUCache(2)\n    c.put(\"a\", 1)\n    c.put(\"b\", 2)\n    assert c.get(\"a\") == 1\n    c.put(\"c\", 3)\n    assert c.get(\"b\") is None and c.keys() == [\"a\", \"c\"]\n    c.put(\"a\", 10)\n    c.put(\"d\", 4)\n    assert c.keys() == [\"a\", \"d\"] and c.get(\"a\") == 10 and c.get(\"c\") is None\ndef test_random():\n    \"against a list\"\n    rng = random.Random(2)\n    c, order, vals = LRUCache(4), [], {}\n    for i in range(3000):\n        k = rng.randrange(8)\n        if rng.random() < 0.5:\n            got = c.get(k)\n            assert got == (vals[k] if k in order else None)\n            if k in order:\n                order.remove(k)\n                order.append(k)\n        else:\n            c.put(k, i)\n            vals[k] = i\n            if k in order:\n                order.remove(k)\n            order.append(k)\n            if len(order) > 4:\n                order.pop(0)\n        assert c.keys() == order\n    big = LRUCache(50_000)\n    t0 = time.perf_counter()\n    for i in range(100_000):\n        big.put(i, i)\n        big.get(i // 2)\n    assert time.perf_counter() - t0 < 3")

### Exercise: B3: Window maximums

Write `window_max(nums, k)`: the maximum of every contiguous window of size k (k ≥ 1; `[]` if k exceeds the length), in O(n) with a deque of indices whose values decrease from front to back. Then `window_ranges(nums, k)`: for every window, its maximum minus its minimum, using two deques.

In [ ]:
from collections import deque

def window_max(nums, k):
    ___

def window_ranges(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_windows():\n    \"windows\"\n    assert window_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7] and window_max([1], 2) == []\n    assert window_ranges([1, 3, -1, -3, 5, 3, 6, 7], 3) == [4, 6, 8, 8, 3, 4]\n    rng = random.Random(3)\n    for _ in range(200):\n        xs = [rng.randint(-9, 9) for _ in range(rng.randint(0, 15))]\n        k = rng.randint(1, 5)\n        ws = [xs[i:i + k] for i in range(len(xs) - k + 1)]\n        assert window_max(xs, k) == [max(w) for w in ws] and window_ranges(xs, k) == [max(w) - min(w) for w in ws]\n    big = list(range(200_000))\n    t0 = time.perf_counter()\n    window_max(big[::-1], 1000)\n    window_ranges(big, 1000)\n    assert time.perf_counter() - t0 < 3")

### Exercise: B4: Saving a tree

For binary trees of `T(v, l, r)` with whole-number values, write `save(root)`: a string of the level-order values separated by commas, with `#` for each missing child of a real node and trailing `#`s removed (`""` for an empty tree), and `load(text)` that rebuilds the tree. No recursion: the checks use a 10,000-node chain.

In [ ]:
from collections import deque

class T:
    def __init__(self, v, l=None, r=None):
        self.v, self.l, self.r = v, l, r

def save(root):
    ___

def load(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef shape(n):\n    out, stack = [], [n]\n    while stack:\n        x = stack.pop()\n        if x is None:\n            out.append(None)\n        else:\n            out.append(x.v)\n            stack += [x.r, x.l]\n    return out\ndef rand_tree(rng, n):\n    if n == 0:\n        return None\n    k = rng.randint(0, n - 1)\n    return T(rng.randint(-20, 20), rand_tree(rng, k), rand_tree(rng, n - 1 - k))\ndef test_save():\n    \"saving\"\n    t = T(1, T(2, None, T(4)), T(3))\n    assert save(t) == \"1,2,3,#,4\" and save(None) == \"\" and load(\"\") is None\n    assert shape(load(\"5,#,7,6\")) == [5, None, 7, 6, None, None, None]\n    rng = random.Random(4)\n    for _ in range(200):\n        t = rand_tree(rng, rng.randint(0, 20))\n        assert shape(load(save(t))) == shape(t)\n    chain = None\n    for v in range(10_000):\n        chain = T(v, chain)\n    assert shape(load(save(chain))) == shape(chain)")

### Exercise: B5: Words in a grid

A grid of letters hides words: a word is formed by moving between horizontally or vertically adjacent cells, using each cell at most once per word. Write `find_words(grid, words)`: the words from the list that appear in the grid, sorted. Put the words in a trie and search from every cell, following trie branches, so prefixes that lead nowhere are abandoned at once. The checks use grids of up to 6 × 6 and 300 words.

In [ ]:
def find_words(grid, words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow_has(grid, w):\n    R, C = len(grid), len(grid[0]) if grid else 0\n    def go(r, c, i, used):\n        if grid[r][c] != w[i]:\n            return False\n        if i == len(w) - 1:\n            return True\n        used.add((r, c))\n        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n            nr, nc = r + dr, c + dc\n            if 0 <= nr < R and 0 <= nc < C and (nr, nc) not in used and go(nr, nc, i + 1, used):\n                used.discard((r, c))\n                return True\n        used.discard((r, c))\n        return False\n    return any(go(r, c, 0, set()) for r in range(R) for c in range(C)) if w else False\ndef test_grid():\n    \"grids\"\n    grid = [\"oaan\", \"etae\", \"ihkr\", \"iflv\"]\n    assert find_words(grid, [\"oath\", \"pea\", \"eat\", \"rain\"]) == [\"eat\", \"oath\"]\n    assert find_words([\"ab\"], [\"aba\", \"ab\", \"ba\"]) == [\"ab\", \"ba\"] and find_words([\"a\"], [\"a\", \"aa\"]) == [\"a\"]\n    rng = random.Random(5)\n    for _ in range(40):\n        R, C = rng.randint(1, 4), rng.randint(1, 4)\n        g = [\"\".join(rng.choice(\"abc\") for _ in range(C)) for _ in range(R)]\n        ws = list({\"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 5))) for _ in range(30)})\n        assert find_words(g, ws) == sorted(w for w in ws if slow_has(g, w)), (g, ws)\n    g = [\"\".join(rng.choice(\"abcde\") for _ in range(6)) for _ in range(6)]\n    ws = list({\"\".join(rng.choice(\"abcde\") for _ in range(rng.randint(3, 8))) for _ in range(300)})\n    t0 = time.perf_counter()\n    find_words(g, ws)\n    assert time.perf_counter() - t0 < 5")

### Bug hunt: B6: Three bugs

`top_words(text, k)` should return the k most common words (lower-cased runs of letters), most common first, ties alphabetical, keeping a heap of at most k entries. It has three bugs: it counts upper- and lower-case forms separately, it keeps the k **least** common words, and it breaks ties the wrong way round. Fix them so this prints `['the', 'and', 'cat']`.

In [ ]:
import heapq, re
from collections import Counter

class Entry:
    def __init__(self, count, word):
        self.count, self.word = count, word
    def __lt__(self, other):
        # "less" means "worse": fewer uses, or the same uses and later in the alphabet
        if self.count != other.count:
            return self.count < other.count
        return self.word < other.word

def top_words(text, k):
    counts = Counter(re.findall(r"[a-zA-Z]+", text))
    heap = []
    for word, count in counts.items():
        e = Entry(count, word)
        if len(heap) < k:
            heapq.heappush(heap, e)
        elif e < heap[0]:
            heapq.heapreplace(heap, e)
    return [e.word for e in sorted(heap, reverse=True)]

print(top_words("The cat and the dog. And the cat sat; a dog ran. THE end", 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['the', 'and', 'cat']\")\n    assert top_words(\"b a c a b c d\", 2) == [\"a\", \"b\"] and top_words(\"\", 3) == []")

## Before you start

**The rules.** You have **2 hours** in total. Work alone. Close your notes, don't use the tutor, and don't search the web: the debrief can only help with gaps it can see. You may run your code and use **Check my answer** as often as you like.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 10 quiz questions | 10 (1 each) |
| B | 6 problems | 90 |
| | **Total** | **100** |

Count a problem's full marks only when every check passes. 60 is a good result; 80 an excellent one.

**Strategy.** Read every problem first and write, for each, the pattern you'd use. Do the ones you're sure of first; don't stay stuck on one for more than 20 minutes. Test small and awkward cases (empty input, one item, ties, duplicates) before pressing Check.

## Part A: quiz

## Part B: six problems

## When you finish

Add up your marks: 1 for each quiz question you got right first time, and the full marks for each problem whose checks all pass. Write your total down, with the time you took, and a note of every problem you didn't finish or found hard. Don't look anything up yet.

Saturday's debrief goes through every question and problem: a model solution, why it works, the mistakes people most often make, and a second chance at each skill. Tomorrow, before that, a different kind of practice: a mock interview, where explaining your thinking matters as much as the code.